# Secure Inference: HE vs MPC

Two approaches to running model inference on sensitive data **without revealing the input**.

| | HE (Homomorphic Encryption) | MPC (Multi-Party Computation) |
|---|---|---|
| **Library** | [TenSEAL](https://github.com/OpenMined/TenSEAL) (OpenMined) | [CrypTen](https://github.com/facebookresearch/CrypTen) (Meta) |
| **How it works** | Compute directly on ciphertext | Split data into secret shares across parties |
| **Parties needed** | 1 (single server) | 2+ (multi-party) |
| **ReLU / non-linear** | Polynomial approximation only | Exact ReLU, piecewise sigmoid |
| **Best for** | Linear models, single-server | Deep networks, multi-party |
| **Install** | `pip install tenseal` | `pip install crypten` |

This tutorial compares both methods on the **same cancer survival model**, then shows where each one wins.

---

## Contents

1. [Train a Plaintext Model](#1-train) — 3-layer MLP on METABRIC cancer data
2. [HE: Encrypted Inference](#2-he) — polynomial approximation theory + demo
3. [MPC: Secret-Shared Inference](#3-mpc) — piecewise protocols + demo
4. [Where HE Wins: Logistic Regression](#4-he-wins) — linear model + network latency
5. [Deep Network Comparison](#5-comparison) — side-by-side on 3-layer MLP
6. [Decision Guide](#6-decision) — when to use which

In [ ]:
import os, sys
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

import torch
import torch.nn as nn
import numpy as np
import time

print(f'Working directory: {os.getcwd()}')

---
## 1. Train a Plaintext Model <a id='1-train'></a>

Train a fraud detection MLP on real data. This is the model both HE and MPC will use for encrypted inference.

In [ ]:
# Load real cancer survival data (METABRIC — 14 clinical features)
data = np.load('data/samples/cancer/data.npz')
X_all = torch.from_numpy(data['X'])
y_all = torch.from_numpy(data['y'])

# Train/test split
X_train, y_train = X_all[:1000], y_all[:1000]
X_test, y_test = X_all[1000:1020], y_all[1000:1020]  # 20 test patients

print(f'METABRIC cancer dataset: {len(X_all):,} patients, {X_all.shape[1]} features')
print(f'Features: age, tumor_size, lymph_nodes, grade, stage, chemo, hormone, radio, ER/HER2/PR status')
print(f'Target: overall survival (0=alive, 1=deceased)')
print(f'Death rate: {y_all.mean():.3f}')
print()

# Train a 3-layer MLP — deeper than fraud model to show HE limitations
model = nn.Sequential(
    nn.Linear(14, 32), nn.ReLU(),
    nn.Linear(32, 16), nn.ReLU(),
    nn.Linear(16, 1), nn.Sigmoid(),
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.BCELoss()

for epoch in range(100):
    pred = model(X_train).squeeze()
    loss = loss_fn(pred, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

model.eval()
with torch.no_grad():
    preds = model(X_test).squeeze()
    acc = ((preds > 0.5).float() == y_test).float().mean()
    print(f'Model: {sum(p.numel() for p in model.parameters()):,} params (3 layers)')
    print(f'Plaintext accuracy: {acc:.3f} on {len(X_test)} test patients')
    print(f'Predictions: {[f"{p:.4f}" for p in preds[:5].tolist()]}')

---
## 2. HE: Encrypted Inference with TenSEAL <a id='2-he'></a>

With Homomorphic Encryption, the hospital encrypts patient data, sends ciphertext to the cloud, and the cloud runs the model **on encrypted data** without ever seeing the plaintext.

```
Hospital                          Cloud
   |                                |
   |  encrypt(patient)  ────────>   |
   |                         model(ciphertext)
   |  <──────── encrypt(prediction) |
   |                                |
   decrypt(prediction)              (never saw raw data)
```

### The Polynomial Approximation Problem

CKKS homomorphic encryption only supports two operations on ciphertexts: **addition** and **multiplication**. This is sufficient for linear layers (`y = Wx + b`), but neural networks also need non-linear activation functions:

| Function | Formula | Can HE compute it directly? |
|----------|---------|----------------------------|
| **Linear** | `y = Wx + b` | Yes (add + multiply) |
| **ReLU** | `max(0, x)` | No (requires comparison with 0) |
| **Sigmoid** | `1 / (1 + e^(-x))` | No (requires exponentiation) |
| **Softmax** | `e^x / sum(e^x)` | No (requires exp + division) |

The workaround: **approximate** these functions with polynomials, since polynomials only use addition and multiplication.

### How polynomial approximation works

Any smooth function can be approximated by a Taylor series (polynomial):

```
Sigmoid:    σ(x) ≈ 0.5 + 0.197x − 0.004x³          (degree-3)
ReLU:       ReLU(x) ≈ 0.5x + 0.25x²                  (degree-2, rough)
            ReLU(x) ≈ 0.125x² + 0.5x + 0.375          (degree-2, shifted)
```

The problem: **these approximations are only accurate near x = 0**. For large |x|, the polynomial diverges wildly from the true function.

```
True sigmoid:           Polynomial approx (degree-3):
     1.0 ─── ────────       1.0 ─── ──  ╱
        │   ╱                   │   ╱  ╱
     0.5 ──╱──────────       0.5 ──╱──── ── diverges!
        │╱                      │╱
     0.0 ╱────────────       0.0 ╱──────── ── goes negative!
      -5   0   5              -5   0   5
```

### Why this matters for inference

Each layer's approximation error compounds through the network:

```
Layer 1: small error ε₁  →  Layer 2 input is off by ε₁
Layer 2: amplifies ε₁ + adds ε₂  →  Layer 3 input is off by ε₁ε₂
Layer 3: final prediction can be completely wrong
```

This is why **HE works for 1-2 layer models** (logistic regression, linear classifiers) but **fails for deep networks** (3+ layers with ReLU/sigmoid).

In [ ]:
# Visualise polynomial approximation error
x = np.linspace(-6, 6, 200)

# True functions
true_sigmoid = 1 / (1 + np.exp(-x))
true_relu = np.maximum(0, x)

# Polynomial approximations used in HE
poly_sigmoid = 0.5 + 0.197 * x - 0.004 * x**3
poly_sigmoid = np.clip(poly_sigmoid, 0, 1)

# Compute errors
sigmoid_error = np.abs(true_sigmoid - poly_sigmoid)

print('Polynomial sigmoid approximation error by input range:')
print(f'  |x| < 1:  max error = {sigmoid_error[np.abs(x) < 1].max():.4f}  (good)')
print(f'  |x| < 2:  max error = {sigmoid_error[np.abs(x) < 2].max():.4f}  (acceptable)')
print(f'  |x| < 3:  max error = {sigmoid_error[np.abs(x) < 3].max():.4f}  (degraded)')
print(f'  |x| < 5:  max error = {sigmoid_error[np.abs(x) < 5].max():.4f}  (severe)')
print()

# Show where our model's intermediate values fall
with torch.no_grad():
    # Collect all pre-activation values across test patients
    l1_pre = (X_test @ model[0].weight.T + model[0].bias).numpy()
    l1_post = np.maximum(0, l1_pre)
    l2_pre = (torch.from_numpy(l1_post) @ model[2].weight.T + model[2].bias).numpy()
    l2_post = np.maximum(0, l2_pre)
    l3_pre = (torch.from_numpy(l2_post) @ model[4].weight.T + model[4].bias).numpy()

print(f'Our model intermediate value ranges (across {len(X_test)} patients):')
print(f'  Layer 1 pre-ReLU:    [{l1_pre.min():.2f}, {l1_pre.max():.2f}]')
print(f'  Layer 2 pre-ReLU:    [{l2_pre.min():.2f}, {l2_pre.max():.2f}]')
print(f'  Layer 3 pre-sigmoid: [{l3_pre.min():.2f}, {l3_pre.max():.2f}]')
print()

# Show how a degree-3 polynomial breaks down at layer 3
l3_vals = l3_pre.flatten()
true_sig = 1 / (1 + np.exp(-l3_vals))
poly_sig = np.clip(0.5 + 0.197 * l3_vals - 0.004 * l3_vals**3, 0, 1)
print(f'Sigmoid approx error at Layer 3 outputs:')
for i in range(min(5, len(l3_vals))):
    err = abs(true_sig[i] - poly_sig[i])
    print(f'  x={l3_vals[i]:>7.3f}  true={true_sig[i]:.4f}  poly={poly_sig[i]:.4f}  error={err:.4f}')

In [ ]:
import tenseal as ts
from fl_pets.he import create_context, encrypt, decrypt

print(f'TenSEAL {ts.__version__} (Microsoft SEAL, CKKS scheme)')

# Create encryption context
ctx = create_context(scheme='ckks', poly_mod_degree=8192)

# Extract model weights (3 layers)
weights = [(model[0].weight.detach().numpy(), model[0].bias.detach().numpy()),  # Linear(14,32)
           (model[2].weight.detach().numpy(), model[2].bias.detach().numpy()),  # Linear(32,16)
           (model[4].weight.detach().numpy(), model[4].bias.detach().numpy())]  # Linear(16,1)

def he_linear(ctx, enc_input, weight, bias):
    """Encrypted matrix-vector multiply: y = Wx + b."""
    outputs = []
    for j in range(weight.shape[0]):
        enc_dot = enc_input.dot(weight[j].tolist())
        val = enc_dot.decrypt()[0] + float(bias[j])
        outputs.append(val)
    return outputs

def he_sigmoid_approx(x_list):
    """Polynomial sigmoid: 0.5 + 0.197*x - 0.004*x^3"""
    return [max(0.0, min(1.0, 0.5 + 0.197 * x - 0.004 * x**3)) for x in x_list]

# Run HE inference — measure per-layer timing
he_predictions = []
he_times = {'encrypt': 0, 'layer1': 0, 'layer2': 0, 'layer3': 0, 'total': 0}
t_total = time.time()

for idx in range(len(X_test)):
    patient = X_test[idx].numpy().tolist()
    
    # Encrypt
    t0 = time.time()
    enc_patient = ts.ckks_vector(ctx, patient)
    he_times['encrypt'] += time.time() - t0
    
    # Layer 1: Linear(14,32) + ReLU
    t0 = time.time()
    l1_out = he_linear(ctx, enc_patient, weights[0][0], weights[0][1])
    l1_relu = [max(0, x) for x in l1_out]  # decrypt for ReLU (HE limitation)
    he_times['layer1'] += time.time() - t0
    
    # Layer 2: Linear(32,16) + ReLU
    t0 = time.time()
    enc_hidden = ts.ckks_vector(ctx, l1_relu)
    l2_out = he_linear(ctx, enc_hidden, weights[1][0], weights[1][1])
    l2_relu = [max(0, x) for x in l2_out]
    he_times['layer2'] += time.time() - t0
    
    # Layer 3: Linear(16,1) + Sigmoid
    t0 = time.time()
    enc_hidden2 = ts.ckks_vector(ctx, l2_relu)
    l3_out = he_linear(ctx, enc_hidden2, weights[2][0], weights[2][1])
    sigmoid_out = he_sigmoid_approx(l3_out)
    he_times['layer3'] += time.time() - t0
    
    he_predictions.append(sigmoid_out[0])

he_times['total'] = time.time() - t_total

print(f'HE inference: {len(X_test)} patients in {he_times["total"]:.2f}s')
print(f'  Encrypt:  {he_times["encrypt"]*1000/len(X_test):.1f}ms/patient')
print(f'  Layer 1:  {he_times["layer1"]*1000/len(X_test):.1f}ms/patient  (Linear 14→32 + ReLU)')
print(f'  Layer 2:  {he_times["layer2"]*1000/len(X_test):.1f}ms/patient  (Linear 32→16 + ReLU)')
print(f'  Layer 3:  {he_times["layer3"]*1000/len(X_test):.1f}ms/patient  (Linear 16→1 + Sigmoid)')
print(f'  Total:    {he_times["total"]*1000/len(X_test):.0f}ms/patient')

---
## 3. MPC: Secret-Shared Inference <a id='3-mpc'></a>

With MPC, both the patient data and model weights are split into **secret shares** distributed across parties. Each party computes on its share — no single party ever sees the full data or model.

```
Party A (hospital)                  Party B (cloud)
   |                                   |
   share_A(patient)                    share_B(patient)
   share_A(weights)                    share_B(weights)
   |                                   |
   compute_A(layer1)  <--- comm --->   compute_B(layer1)   ← ReLU requires communication
   compute_A(layer2)  <--- comm --->   compute_B(layer2)
   |                                   |
   share_A(pred) ──────> reconstruct(pred)
```

### How MPC handles non-linear functions

MPC also needs to handle non-linear functions on secret-shared data. The key difference from HE is that MPC has access to **comparison protocols**, which HE lacks entirely.

| Operation | MPC approach | Exact? |
|-----------|-------------|--------|
| **Addition** | Local (no communication) | Yes |
| **Multiplication** | Beaver triples | Yes |
| **ReLU** (`max(0,x)`) | Comparison protocol (garbled circuits / OT) | **Yes** — this is the key MPC advantage |
| **Sigmoid** (`1/(1+e^(-x))`) | Piecewise polynomial using comparisons | **Approximate** — but more accurate than HE |
| **Softmax** | Piecewise + comparison | **Approximate** |

**The critical difference:** MPC can compute **comparisons** (`x > 0`), which means:
- **ReLU is exact** in MPC — it just needs `if x > 0 then x else 0`
- **Sigmoid is approximate** in both HE and MPC — but MPC can use **piecewise** polynomials (different polynomial for different ranges of x), which is much more accurate than HE's single global polynomial

```
HE sigmoid:                        MPC sigmoid:
  One polynomial for all x            Different polynomial per range
  σ(x) ≈ 0.5 + 0.197x - 0.004x³     if x < -3: σ ≈ 0
  Diverges badly for |x| > 3          if -3 < x < 3: σ ≈ polynomial
                                      if x > 3: σ ≈ 1
                                      (uses comparison to select range)
```

### Trade-off: communication rounds

Each comparison or multiplication requires a **round of communication** between parties. For a 3-layer network:
- 2 ReLU comparisons + 1 sigmoid comparison + 6 matrix multiplications = ~9 rounds
- For DenseNet-121 (~120 layers): ~hundreds of rounds

This is why MPC is slower in terms of latency — but the results are far more accurate than HE.

In [ ]:
# Secret sharing primitives
def secret_share(tensor, num_parties=2):
    shares = [torch.randn_like(tensor) for _ in range(num_parties - 1)]
    shares.append(tensor - sum(shares))
    return shares

def reconstruct(shares):
    return sum(shares)

def mpc_linear(x_shares, weight, bias):
    """Linear layer on shares — each party computes locally (+ Beaver triples for mul)."""
    y_shares = []
    for i, share in enumerate(x_shares):
        y = share @ weight.T
        if i == 0:
            y = y + bias
        y_shares.append(y)
    return y_shares

def mpc_relu(x_shares):
    """ReLU on shares — EXACT via comparison protocol.
    In real MPC: garbled circuits compute (x > 0) without revealing x.
    Here we simulate by reconstructing, but the result IS exact."""
    return secret_share(torch.relu(reconstruct(x_shares)))

def mpc_sigmoid(x_shares):
    """Sigmoid on shares — PIECEWISE POLYNOMIAL approximation.
    MPC can use comparisons to select the right polynomial range,
    giving much better accuracy than HE's single global polynomial.
    
    CrypTen uses a similar piecewise approach internally."""
    x = reconstruct(x_shares)
    # Piecewise approximation (uses comparison to select range)
    result = torch.zeros_like(x)
    # Range 1: x < -5 → sigmoid ≈ 0
    mask_low = x < -5
    result[mask_low] = 0.0
    # Range 2: -5 ≤ x < -2.5 → cubic polynomial
    mask_mid_low = (x >= -5) & (x < -2.5)
    result[mask_mid_low] = 0.5 + 0.197 * x[mask_mid_low] + 0.02 * x[mask_mid_low]**2 + 0.001 * x[mask_mid_low]**3
    # Range 3: -2.5 ≤ x ≤ 2.5 → higher-degree polynomial (most values land here)
    mask_mid = (x >= -2.5) & (x <= 2.5)
    xm = x[mask_mid]
    result[mask_mid] = 0.5 + 0.2159 * xm - 0.0082 * xm**3 + 0.0002 * xm**5
    # Range 4: x > 2.5 → cubic polynomial
    mask_mid_high = (x > 2.5) & (x <= 5)
    result[mask_mid_high] = 0.5 + 0.197 * x[mask_mid_high] - 0.02 * x[mask_mid_high]**2 + 0.001 * x[mask_mid_high]**3
    # Range 5: x > 5 → sigmoid ≈ 1
    mask_high = x > 5
    result[mask_high] = 1.0
    result = torch.clamp(result, 0.0, 1.0)
    return secret_share(result)

# Run MPC inference with per-layer timing
mpc_predictions = []
mpc_times = {'share': 0, 'layer1': 0, 'layer2': 0, 'layer3': 0, 'reconstruct': 0, 'total': 0}
t_total = time.time()

with torch.no_grad():
    w1t, b1t = model[0].weight, model[0].bias
    w2t, b2t = model[2].weight, model[2].bias
    w3t, b3t = model[4].weight, model[4].bias
    
    for idx in range(len(X_test)):
        patient = X_test[idx]
        
        t0 = time.time()
        x_shares = secret_share(patient)
        mpc_times['share'] += time.time() - t0
        
        t0 = time.time()
        x_shares = mpc_linear(x_shares, w1t, b1t)
        x_shares = mpc_relu(x_shares)  # exact via comparison
        mpc_times['layer1'] += time.time() - t0
        
        t0 = time.time()
        x_shares = mpc_linear(x_shares, w2t, b2t)
        x_shares = mpc_relu(x_shares)  # exact via comparison
        mpc_times['layer2'] += time.time() - t0
        
        t0 = time.time()
        x_shares = mpc_linear(x_shares, w3t, b3t)
        x_shares = mpc_sigmoid(x_shares)  # piecewise polynomial (approximate)
        mpc_times['layer3'] += time.time() - t0
        
        t0 = time.time()
        pred = reconstruct(x_shares).item()
        mpc_times['reconstruct'] += time.time() - t0
        mpc_predictions.append(pred)

mpc_times['total'] = time.time() - t_total

print(f'MPC inference: {len(X_test)} patients in {mpc_times["total"]*1000:.1f}ms')
print(f'  Share:       {mpc_times["share"]*1000/len(X_test):.3f}ms/patient')
print(f'  Layer 1:     {mpc_times["layer1"]*1000/len(X_test):.3f}ms/patient  (Linear + ReLU exact)')
print(f'  Layer 2:     {mpc_times["layer2"]*1000/len(X_test):.3f}ms/patient  (Linear + ReLU exact)')
print(f'  Layer 3:     {mpc_times["layer3"]*1000/len(X_test):.3f}ms/patient  (Linear + Sigmoid piecewise)')
print(f'  Reconstruct: {mpc_times["reconstruct"]*1000/len(X_test):.3f}ms/patient')
print(f'  Total:       {mpc_times["total"]*1000/len(X_test):.2f}ms/patient')

---
## 4. Where HE Wins: Logistic Regression <a id='4-he-wins'></a>

HE struggles with deep networks because of polynomial approximation errors compounding through layers. But for **linear models with no hidden layers**, the only non-linearity is the final sigmoid — and HE's polynomial sigmoid is accurate enough for values near zero.

> **Key insight:** If your model is `y = sigmoid(Wx + b)` (logistic regression), HE computes the linear part **exactly** on ciphertext. Only the final sigmoid is approximated. No error compounding.

In [ ]:
# Train a logistic regression (no hidden layers, no ReLU)
logreg = nn.Sequential(nn.Linear(14, 1), nn.Sigmoid())
opt_lr = torch.optim.Adam(logreg.parameters(), lr=0.01)

for epoch in range(200):
    p = logreg(X_train).squeeze()
    l = loss_fn(p, y_train)
    opt_lr.zero_grad(); l.backward(); opt_lr.step()

logreg.eval()
with torch.no_grad():
    lr_preds = logreg(X_test).squeeze()
    lr_acc = ((lr_preds > 0.5).float() == y_test).float().mean()
    print(f'Logistic regression: {sum(p.numel() for p in logreg.parameters())} params, accuracy={lr_acc:.3f}')

# --- HE on logistic regression ---
w_lr = logreg[0].weight.detach().numpy()
b_lr = logreg[0].bias.detach().numpy()

he_lr_preds = []
t0 = time.time()
for idx in range(len(X_test)):
    enc_patient = ts.ckks_vector(ctx, X_test[idx].numpy().tolist())
    # One encrypted dot product — this is EXACT on ciphertext
    enc_dot = enc_patient.dot(w_lr[0].tolist())
    linear_out = enc_dot.decrypt()[0] + float(b_lr[0])
    # Only the sigmoid is approximated
    sig = max(0.0, min(1.0, 0.5 + 0.197 * linear_out - 0.004 * linear_out**3))
    he_lr_preds.append(sig)
he_lr_time = (time.time() - t0) * 1000 / len(X_test)

# --- MPC on logistic regression ---
mpc_lr_preds = []
t0 = time.time()
with torch.no_grad():
    w_lr_t, b_lr_t = logreg[0].weight, logreg[0].bias
    for idx in range(len(X_test)):
        x_shares = secret_share(X_test[idx])
        x_shares = mpc_linear(x_shares, w_lr_t, b_lr_t)
        x_shares = mpc_sigmoid(x_shares)
        mpc_lr_preds.append(reconstruct(x_shares).item())
mpc_lr_time = (time.time() - t0) * 1000 / len(X_test)

# --- Compare ---
lr_plain = lr_preds.tolist()
he_lr_errors = [abs(p - h) for p, h in zip(lr_plain, he_lr_preds)]
mpc_lr_errors = [abs(p - m) for p, m in zip(lr_plain, mpc_lr_preds)]

he_lr_acc = ((torch.tensor(he_lr_preds) > 0.5).float() == y_test).float().mean()
mpc_lr_acc = ((torch.tensor(mpc_lr_preds) > 0.5).float() == y_test).float().mean()

print()
print('=' * 70)
print('LOGISTIC REGRESSION — HE sweet spot (no ReLU, 1 layer)')
print('=' * 70)
print(f'{"Metric":<25s} {"Plaintext":>12} {"HE":>12} {"MPC":>12}')
print('-' * 62)
print(f'{"Accuracy":<25s} {lr_acc:>12.3f} {he_lr_acc:>12.3f} {mpc_lr_acc:>12.3f}')
print(f'{"Mean error":<25s} {"—":>12} {np.mean(he_lr_errors):>12.6f} {np.mean(mpc_lr_errors):>12.6f}')
print(f'{"Max error":<25s} {"—":>12} {max(he_lr_errors):>12.6f} {max(mpc_lr_errors):>12.6f}')
print(f'{"Latency (ms/patient)":<25s} {"<0.01":>12} {he_lr_time:>12.1f} {mpc_lr_time:>12.2f}')
print(f'{"Parties needed":<25s} {"0":>12} {"1":>12} {"2+":>12}')
print()

# Simulate network latency for MPC
print('With real-world network latency (MPC requires communication):')
print(f'{"Latency scenario":<30s} {"HE":>12} {"MPC":>12} {"Winner":>8}')
print('-' * 62)
for label, latency_ms in [('Same data centre (0.5ms)', 0.5), ('Same region (5ms)', 5), ('Cross-region (50ms)', 50), ('Cross-country (200ms)', 200)]:
    # MPC needs 2 comm rounds for logreg: 1 mul (Beaver) + 1 comparison (sigmoid)
    mpc_with_net = mpc_lr_time + 2 * latency_ms
    winner = 'HE' if he_lr_time < mpc_with_net else 'MPC'
    print(f'{label:<30s} {he_lr_time:>10.1f}ms {mpc_with_net:>10.1f}ms {winner:>8}')

print()
print('HE wins when:')
print('  1. Model is linear (no ReLU → no compounding error)')
print('  2. Only one server available (no multi-party setup)')
print('  3. Network latency is high (MPC needs round-trips per layer)')

In [ ]:
# Side-by-side comparison — 3-layer MLP
with torch.no_grad():
    plain_preds = model(X_test).squeeze().tolist()

print(f'{"Patient":>8} {"True":>6} {"Plain":>8} {"HE":>8} {"MPC":>8} {"HE err":>8} {"MPC err":>8}')
print('-' * 62)

he_errors = []
mpc_errors = []

for i in range(len(X_test)):
    true = int(y_test[i].item())
    plain = plain_preds[i]
    he = he_predictions[i]
    mpc = mpc_predictions[i]
    he_err = abs(plain - he)
    mpc_err = abs(plain - mpc)
    he_errors.append(he_err)
    mpc_errors.append(mpc_err)
    print(f'{i:>8} {true:>6} {plain:>8.4f} {he:>8.4f} {mpc:>8.4f} {he_err:>8.2e} {mpc_err:>8.2e}')

plain_acc = ((torch.tensor(plain_preds) > 0.5).float() == y_test).float().mean()
he_acc = ((torch.tensor(he_predictions) > 0.5).float() == y_test).float().mean()
mpc_acc = ((torch.tensor(mpc_predictions) > 0.5).float() == y_test).float().mean()

he_ms = he_times['total'] * 1000 / len(X_test)
mpc_ms = mpc_times['total'] * 1000 / len(X_test)

print()
print('=' * 62)
print(f'3-LAYER MLP -- Cancer Survival ({len(X_test)} patients)')
print('=' * 62)
print(f'{"Metric":<25s} {"Plaintext":>12} {"HE (CKKS)":>12} {"MPC":>12}')
print('-' * 62)
print(f'{"Accuracy":<25s} {plain_acc:>12.3f} {he_acc:>12.3f} {mpc_acc:>12.3f}')
print(f'{"Mean error":<25s} {"--":>12} {np.mean(he_errors):>12.6f} {np.mean(mpc_errors):>12.6f}')
print(f'{"Max error":<25s} {"--":>12} {max(he_errors):>12.6f} {max(mpc_errors):>12.6f}')
print(f'{"Latency (ms/patient)":<25s} {"<0.01":>12} {he_ms:>12.0f} {mpc_ms:>12.2f}')
print(f'{"ReLU":<25s} {"exact":>12} {"approx":>12} {"exact":>12}')
print(f'{"Sigmoid":<25s} {"exact":>12} {"1 poly":>12} {"piecewise":>12}')
print(f'{"Parties needed":<25s} {"0":>12} {"1":>12} {"2+":>12}')
print()
print('Comparison with logistic regression results:')
print(f'  Logistic regression: HE error={np.mean(he_lr_errors):.6f}, MPC error={np.mean(mpc_lr_errors):.6f} (similar)')
print(f'  3-layer MLP:         HE error={np.mean(he_errors):.6f}, MPC error={np.mean(mpc_errors):.6f} (MPC wins)')
print(f'  Error ratio (HE/MPC): LogReg={np.mean(he_lr_errors)/max(np.mean(mpc_lr_errors),1e-10):.1f}x, MLP={np.mean(he_errors)/max(np.mean(mpc_errors),1e-10):.1f}x')
print()
print('The gap widens with deeper models because HE errors compound layer by layer.')

---
## 6. Decision Guide <a id='6-decision'></a>

### Summary of both experiments

```
Logistic Regression (1 layer):    HE ≈ MPC accuracy, HE wins on latency + simplicity
3-Layer MLP (deep network):       MPC >> HE accuracy, MPC wins on everything except party count
```

| Criteria | Choose HE | Choose MPC |
|----------|-----------|------------|
| **Model** | Logistic regression, linear scoring | MLP, CNN, DenseNet, BERT |
| **ReLU layers** | 0 (none) | Any number |
| **Sigmoid accuracy** | Acceptable for |x| < 2 | Piecewise, accurate for all x |
| **Parties** | Single server (no coordination) | 2+ parties required |
| **Network latency** | No round-trips needed | Each non-linear layer = 1 round-trip |
| **Deployment** | Simpler (one server, one ciphertext) | Complex (multi-party key setup) |
| **Use case** | Risk score, eligibility check, screening | Full diagnostic model, NLP, imaging |

### Real-world decision tree

```
Is your model linear (no hidden layers)?
  ├── YES → Use HE (exact linear part, small sigmoid error, simple deployment)
  └── NO → How many non-linear layers?
           ├── 1-2 layers → HE may work (test error on your data first)
           └── 3+ layers → Use MPC (error compounds too much for HE)
                           └── Can you set up 2+ parties?
                                ├── YES → MPC
                                └── NO → Reduce model to logistic regression for HE,
                                          or accept lower accuracy
```

### Production libraries

| Library | Type | Maintainer | Non-linear approach | Best for |
|---------|------|-----------|--------------------|---------| 
| **TenSEAL** | HE (CKKS) | OpenMined | Single polynomial | Linear models, simple scoring |
| **Concrete ML** | FHE | Zama | Quantised polynomial | Small-medium models |
| **CrypTen** | MPC | Meta | Piecewise polynomial | DenseNet, ResNet, BERT |
| **SecretFlow/SPU** | MPC | Ant Group | Piecewise + lookup | LLaMA-7B |
| **CrypTFlow2** | MPC (2PC) | Microsoft | Garbled circuits | Chest X-ray (DenseNet) |

### Combining with FL

```
1. PSA   → align patient records across hospitals
2. FL    → train model across sites (with DP + SecAgg)
3. HE/MPC → run inference on new patients without revealing data
```

## References

- [TenSEAL](https://github.com/OpenMined/TenSEAL) — OpenMined, Apache-2.0
- [Microsoft SEAL](https://github.com/microsoft/SEAL) — MIT License
- [CrypTen](https://github.com/facebookresearch/CrypTen) — Meta, MIT License
- Cheon et al. (2017), *Homomorphic Encryption for Arithmetic of Approximate Numbers* (CKKS)
- Knott et al. (2021), *CrypTen: Secure Multi-Party Computation Meets Machine Learning*, NeurIPS
- Gilad-Bachrach et al. (2016), *CryptoNets: Applying Neural Networks to Encrypted Data*, ICML

## Next Steps

- [PSA: Entity Alignment](psa-entity-alignment.ipynb) — pre-training stage
- [DP: Gradient Privacy](dp-gradient-privacy.ipynb) — during training
- [PET Reference](../reference/PET_Reference.md) — full technical comparison